In [5]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Načte proměnné ze souboru .env v kořenové složce projektu
load_dotenv()

# Kořenová složka projektu (složka, ve které je tento skript uložený)
PROJECT_DIR = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd().parent

raw_dir = PROJECT_DIR / "data" / "raw"
db_dir = PROJECT_DIR / "data" / "db"
src_dir = PROJECT_DIR / "src"

# Vytvoření složek (funguje stejně na Windows, macOS i Linuxu)
raw_dir.mkdir(parents=True, exist_ok=True)  # sem vlož PDF skripta
db_dir.mkdir(parents=True, exist_ok=True)    # sem se uloží vektorová DB
src_dir.mkdir(parents=True, exist_ok=True)  # sem půjdou .py moduly

# Vyhledání PDF souborů (chyběl ti také import glob, pathlib to řeší rovnou)
pdf_files = list(raw_dir.glob("*.pdf"))
print(f"Nalezeno PDF souborů: {len(pdf_files)}")

# Ověření načtení OpenAI klíče ze souboru .env
if os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY úspěšně načten.")
else:
    print("POZOR: OPENAI_API_KEY nebyl nalezen v souboru .env!")

print("Prostředí připraveno v:", PROJECT_DIR)

Nalezeno PDF souborů: 2
POZOR: OPENAI_API_KEY nebyl nalezen v souboru .env!
Prostředí připraveno v: /home/b1aza/School/SIS/Semestralka


In [ ]:
# !pip install -q openai sympy pymupdf4llm chromadb streamlit

In [6]:
import os
import json
import sympy as sp
from openai import OpenAI

# API klíč se načítá ze souboru .env (viz první buňka s load_dotenv())

# Automaticky načte os.environ["OPENAI_API_KEY"]
client = OpenAI()

# 1. Tvá deterministická funkce pro ověření (běží lokálně v Pythonu)
def verify_equivalence(student_expr: str, expected_expr: str, variable: str = "x") -> str:
    """Ověří matematickou ekvivalenci dvou výrazů pomocí SymPy."""
    try:
        x = sp.Symbol(variable)
        diff = sp.simplify(sp.sympify(student_expr) - sp.sympify(expected_expr))
        return json.dumps({"is_correct": bool(diff == 0), "difference": str(diff)})
    except Exception as e:
        return json.dumps({"is_correct": False, "error": str(e)})

# 2. Popis nástroje pro OpenAI
tools = [
    {
        "type": "function",
        "function": {
            "name": "verify_equivalence",
            "description": "Ověří pomocí SymPy, zda jsou dva matematické výrazy ekvivalentní.",
            "parameters": {
                "type": "object",
                "properties": {
                    "student_expr": {
                        "type": "string",
                        "description": "Výraz od studenta v syntaxi Python/SymPy (např. 'cos(2*x)')"
                    },
                    "expected_expr": {
                        "type": "string",
                        "description": "Správný referenční výraz v syntaxi SymPy (např. '2*cos(2*x)')"
                    },
                    "variable": {
                        "type": "string",
                        "description": "Proměnná, výchozí 'x'"
                    }
                },
                "required": ["student_expr", "expected_expr"]
            }
        }
    }
]

messages = [
    {
        "role": "system",
        "content": "Jsi AI tutor matematické analýzy na FSI VUT. Nikdy nepočítej z hlavy, vždy použij nástroj verify_equivalence."
    },
    {
        "role": "user",
        "content": "Student tvrdí, že derivace sin(2*x) je cos(2*x). Ověř to nástrojem a napiš mu krátkou nápovědu bez prozrazení výsledku."
    }
]

# První volání: model se rozhodne zavolat SymPy
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools
)

msg = response.choices[0].message
messages.append(msg)

# Vykonání lokální Python funkce a vrácení výsledku modelu
if msg.tool_calls:
    for tool_call in msg.tool_calls:
        args = json.loads(tool_call.function.arguments)
        result = verify_equivalence(**args)
        print("Výstup ze SymPy:", result)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })

    # Druhé volání: model zformuluje pedagogickou odpověď podle výsledku ze SymPy
    final_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    print("\nOdpověď studentovi:\n", final_response.choices[0].message.content)

Výstup ze SymPy: {"is_correct": false, "difference": "-cos(2*x)"}

Odpověď studentovi:
 Ahoj, rád bych ti pomohl s tvým tvrzením o derivaci funkce \( \sin(2x) \). Zkus si připomenout, jaký je vztah mezi derivací složené funkce a základními derivacemi trigonometrických funkcí. Vzpomínej také na pravidlo pro derivaci složené funkce (řetězové pravidlo). Měj na paměti, že v této situaci bude důležité zohlednit koeficient, který se může objevit. Držím palce!


In [7]:
import pymupdf4llm
import glob

pdf_files = glob.glob(f"{PROJECT_DIR}/data/raw/*.pdf")
for pdf in pdf_files:
    md_text = pymupdf4llm.to_markdown(pdf)
    out_path = pdf.replace("/raw/", "/").replace(".pdf", ".md")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(md_text)
    print("Převedeno:", out_path)


=== Document parser messages ===
Převedeno: /home/b1aza/School/SIS/Semestralka/data/mat-analyza-i-b5-2024-09-16.md

=== Document parser messages ===
Převedeno: /home/b1aza/School/SIS/Semestralka/data/mat-analyza-ii-b5-2025-02-21.md


In [1]:
%%writefile math_core.py
"""Matematické jádro – deterministický výpočet a ověřování pomocí SymPy.

Modul poskytuje dva nástroje, které orchestrační vrstva (`app.py`) nabízí
jazykovému modelu jako takzvané „tools“. Díky nim model nepočítá zpaměti,
ale výsledky dostává z klasické symbolické matematiky:

- ``verify_equivalence`` – ověří, zda jsou dva výrazy algebraicky ekvivalentní,
- ``compute``             – přímo spočítá derivaci, integrál, limitu, úpravu apod.
"""

import sympy as sp


def verify_equivalence(student_expr: str, expected_expr: str, variable: str = "x") -> dict:
    """Ověří, zda jsou dva výrazy algebraicky ekvivalentní (rozdíl se zjednoduší na 0).

    Args:
        student_expr: výraz studenta v syntaxi SymPy, např. ``"cos(2*x)"``.
        expected_expr: správný referenční výraz, např. ``"2*cos(2*x)"``.
        variable: proměnná, ve které se porovnává (výchozí ``"x"``).

    Returns:
        dict s ``is_correct`` a ``difference`` (nebo ``error`` při chybě zápisu).
    """
    try:
        x = sp.Symbol(variable)
        diff = sp.simplify(sp.sympify(student_expr) - sp.sympify(expected_expr))
        return {"is_correct": bool(diff == 0), "difference": str(diff)}
    except Exception as e:
        return {"is_correct": False, "error": str(e)}


# Mapování (i českých) názvů operací na interní klíč
_OPERATION_ALIASES = {
    "simplify": "simplify", "zjednodusit": "simplify", "upravit": "simplify",
    "derivative": "derivative", "derivace": "derivative", "diff": "derivative",
    "derivative2": "derivative2", "druha_derivace": "derivative2", "diff2": "derivative2",
    "integrate": "integrate", "integral": "integrate", "primitivni_funkce": "integrate",
    "limit": "limit", "limita": "limit",
    "expand": "expand", "rozvinout": "expand",
    "factor": "factor", "faktorizovat": "factor",
    "solve": "solve", "resit": "solve", "koreny": "solve",
}


def _parse_point(point) -> sp.Expr:
    """Převede bod (i nekonečno) na SymPy výraz."""
    p = str(point).strip().lower()
    if p in ("oo", "+oo", "inf", "+inf", "infinity", "nekonecno", "+nekonecno"):
        return sp.oo
    if p in ("-oo", "-inf", "-infinity", "-nekonecno"):
        return -sp.oo
    return sp.sympify(point)


def compute(expr: str, operation: str, variable: str = "x", point=None) -> dict:
    """Přímo spočítá požadovanou operaci nad výrazem pomocí SymPy.

    Args:
        expr: vstupní výraz v syntaxi SymPy, např. ``"sin(2*x)"``.
        operation: ``simplify``, ``derivative``, ``derivative2``, ``integrate``,
            ``limit``, ``expand``, ``factor`` nebo ``solve`` (lze i česky).
        variable: proměnná výpočtu (výchozí ``"x"``).
        point: bod pro ``limit`` (např. ``0``, ``"oo"``, ``"-oo"``).

    Returns:
        dict s ``result`` (textově), ``latex`` a ``operation``;
        při chybě ``result=None`` a ``error``.
    """
    try:
        x = sp.Symbol(variable)
        e = sp.sympify(expr)
        op = _OPERATION_ALIASES.get(str(operation).strip().lower())
        if op is None:
            return {"result": None, "error": f"Neznámá operace '{operation}'."}

        if op == "simplify":
            res = sp.simplify(e)
        elif op == "derivative":
            res = sp.diff(e, x)
        elif op == "derivative2":
            res = sp.diff(e, x, 2)
        elif op == "integrate":
            res = sp.integrate(e, x)
        elif op == "expand":
            res = sp.expand(e)
        elif op == "factor":
            res = sp.factor(e)
        elif op == "solve":
            res = sp.solve(e, x)
        elif op == "limit":
            if point is None:
                return {"result": None, "error": "Pro operaci 'limit' je třeba zadat 'point'."}
            res = sp.limit(e, x, _parse_point(point))
        else:  # pragma: no cover – ošetřeno výše
            return {"result": None, "error": f"Operace '{op}' není podporována."}

        try:
            latex = sp.latex(res)
        except Exception:
            latex = str(res)
        return {"result": str(res), "latex": latex, "operation": op}
    except Exception as e:
        return {"result": None, "error": str(e)}


Writing math_core.py


In [3]:
%%writefile curriculum.py
CURRICULUM = {
    "1. Množiny a reálná čísla": [
        "Základy výrokové logiky a teorie množin",
        "Zobrazení, prostá a inverzní funkce",
        "Reálná čísla, ohraničenost, supremum a infimum",
        "Absolutní hodnota, intervaly a okolí bodu",
    ],
    "2. Funkce jedné proměnné, limita a spojitost": [
        "Definiční obor, obor hodnot a vlastnosti funkcí",
        "Elementární a cyklometrické funkce",
        "Vlastní a nevlastní limita funkce",
        "Spojitost funkce, body nespojitosti a věty o spojitých funkcích",
    ],
    "3. Diferenciální počet funkce jedné proměnné": [
        "Derivace funkce (složená a inverzní funkce, vyšší řády)",
        "Tečna, normála a diferenciál funkce",
        "Věty o střední hodnotě a L'Hospitalovo pravidlo",
        "Monotonnost, lokální a globální extrémy",
        "Konvexnost, konkávnost, inflexní body a asymptoty",
        "Kompletní vyšetření průběhu funkce",
        "Taylorův polynom a odhad zbytku",
    ],
    "4. Integrální počet funkce jedné proměnné": [
        "Primitivní funkce, základní vzorce a úpravy integrandu",
        "Integrace metodou per partes a substituční metodou",
        "Integrace racionálních, goniometrických a iracionálních funkcí",
        "Určitý Riemannův integrál a Newtonova–Leibnizova formule",
        "Aplikace určitého integrálu (obsah, délka oblouku, objem a povrch)",
        "Nevlastní integrály 1. a 2. druhu",
    ],
    "5. Metrické prostory": [
        "Metrika, vzdálenost a koule v metrickém prostoru",
        "Vnitřek, hranice a uzávěr množiny, otevřené a uzavřené množiny",
        "Konvergence posloupností a cauchyovské posloupnosti",
        "Úplné a kompaktní metrické prostory",
        "Spojitost a stejnoměrná spojitost zobrazení, kontrakce a pevný bod",
    ],
    "6. Diferenciální počet funkce více proměnných": [
        "Definiční obor, limita a spojitost funkce více proměnných",
        "Parciální derivace, směrová derivace a gradient",
        "Totální diferenciál, tečná rovina a normála",
        "Derivace složené funkce a Taylorův polynom více proměnných",
        "Volné lokální extrémy a Hessova matice",
        "Implicitní funkce, Jacobiova matice a jakobián",
        "Vázané extrémy (Lagrangeovy multiplikátory) a globální extrémy",
    ],
    "7. Integrální počet funkce více proměnných": [
        "Dvojný integrál a Fubiniova věta na normální množině",
        "Trojný integrál v prostoru",
        "Substituce ve dvojném a trojném integrálu (polární, cylindrické, sférické souřadnice)",
        "Aplikace v mechanice (obsah, objem, hmotnost, těžiště, momenty setrvačnosti)",
    ],
    "8. Křivkové a plošné integrály": [
        "Křivky v rovině a prostoru, parametrizace a tečný vektor",
        "Křivkový integrál I. a II. druhu, Greenova věta",
        "Nezávislost křivkového integrálu na cestě a potenciálové pole",
        "Plochy v prostoru, normálový vektor a orientace plochy",
        "Plošný integrál I. a II. druhu, Gaussova–Ostrogradského a Stokesova věta",
    ],
}

In [ ]:
%%writefile rag.py
"""RAG (Retrieval-Augmented Generation) nad přednáškovými texty.

Modul zajišťuje tzv. znalostní vrstvu aplikace:

1. Zpracuje markdown soubory vzniklé převodem přednáškových PDF
   (``data/mat-analyza-*.md``) a rozseká je na menší pasáže (chunky).
2. Pasáže oboduje pomocí OpenAI embeddings a uloží do vektorové databáze
   ChromaDB (persistentně v ``data/db``).
3. Pro dotaz studenta vrátí nejpodobnější pasáže, které se vloží do
   systémového promptu LLM. Tím se odpovědi opírají o reálné přednášky
   místo o „vymýšlení" modelu.

Veřejné rozhraní:
    build_index()      -- vytvoří / přegeneruje vektorovou DB,
    load_retriever()   -- načte hotový index (nebo vrátí None, pokud chybí),
    Retriever.retrieve -- vyhledá relevantní pasáže k dotazu,
    retrieve()         -- pohodlný obal nad load_retriever().
"""

from __future__ import annotations

import os
import re
from pathlib import Path
from typing import Dict, List, Optional

import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

# --- Konfigurace ---
COLLECTION_NAME = "mat_analyza"
EMBEDDING_MODEL = "text-embedding-3-small"
MAX_CHARS = 1500  # maximální délka jedné pasáže
OVERLAP = 250     # přesah mezi dělenými dlouhými odstavci

PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"
DB_PATH = DATA_DIR / "db"

_HEADING_RE = re.compile(r"^(#{1,6})\s+(.*)$")


# ---------------------------------------------------------------------------
# Dělení textu na pasáže
# ---------------------------------------------------------------------------
def _split_sections(text: str) -> List[tuple]:
    """Rozdělí markdown na sekce podle nadpisů. Vrací seznam (nadpis, tělo)."""
    sections: List[tuple] = []
    current_heading = ""
    buf: List[str] = []
    for line in text.splitlines():
        if _HEADING_RE.match(line.strip()):
            if buf:
                sections.append((current_heading, "\n".join(buf).strip()))
                buf = []
            current_heading = line.strip()
        else:
            buf.append(line)
    if buf:
        sections.append((current_heading, "\n".join(buf).strip()))
    return sections


def chunk_markdown(text: str, source: str, max_chars: int = MAX_CHARS,
                   overlap: int = OVERLAP) -> List[Dict]:
    """Rozseká text na pasáže.

    Každá pasáž si sebou nese nadpis sekce (kvůli kontextu) a název zdrojového
    souboru. Dlouhé sekce se dělí po odstavcích, extrémně dlouhé odstavce po
    znacích s mírným přesahem.
    """
    chunks: List[Dict] = []

    def add(heading: str, body: str) -> None:
        body = body.strip()
        if not body:
            return
        content = f"{heading}\n\n{body}".strip() if heading else body
        idx = len(chunks)
        chunks.append({
            "id": f"{source}::{idx}",
            "text": content,
            "source": source,
            "heading": heading,
        })

    for heading, body in _split_sections(text):
        if not body:
            if heading:
                add("", heading)
            continue

        paragraphs = [p.strip() for p in re.split(r"\n\s*\n", body) if p.strip()]
        buffer = ""

        def flush() -> None:
            nonlocal buffer
            if buffer.strip():
                add(heading, buffer)
            buffer = ""

        for para in paragraphs:
            if len(para) > max_chars:
                flush()
                start = 0
                step = max(max_chars - overlap, 1)
                while start < len(para):
                    add(heading, para[start:start + max_chars])
                    start += step
            elif len(buffer) + len(para) + 2 <= max_chars:
                buffer = f"{buffer}\n\n{para}".strip()
            else:
                flush()
                buffer = para
        flush()

    return chunks


# ---------------------------------------------------------------------------
# ChromaDB
# ---------------------------------------------------------------------------
def _embedding_function() -> OpenAIEmbeddingFunction:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError(
            "OPENAI_API_KEY není nastaven. Vytvoř soubor .env s klíčem "
            "OPENAI_API_KEY=... (viz README)."
        )
    return OpenAIEmbeddingFunction(api_key=api_key, model_name=EMBEDDING_MODEL)


def _client() -> chromadb.ClientAPI:
    DB_PATH.mkdir(parents=True, exist_ok=True)
    return chromadb.PersistentClient(path=str(DB_PATH))


def _default_md_paths() -> List[Path]:
    return sorted(DATA_DIR.glob("mat-analyza-*.md"))


def build_index(md_paths: Optional[List[Path]] = None, rebuild: bool = False) -> int:
    """Vytvoří vektorový index z markdown souborů. Vrací počet pasáží.

    Args:
        md_paths: seznam souborů; pokud None, vezme všechny ``data/mat-analyza-*.md``.
        rebuild: True = existující index smaže a postaví znovu.
    """
    paths = md_paths if md_paths is not None else _default_md_paths()
    if not paths:
        raise FileNotFoundError(f"Nenalezeny žádné markdown soubory v {DATA_DIR}")

    client = _client()

    if rebuild:
        try:
            client.delete_collection(COLLECTION_NAME)
        except Exception:
            pass

    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        embedding_function=_embedding_function(),
    )

    if collection.count() > 0 and not rebuild:
        return collection.count()

    all_chunks: List[Dict] = []
    for path in paths:
        text = Path(path).read_text(encoding="utf-8")
        all_chunks.extend(chunk_markdown(text, source=Path(path).name))

    BATCH = 100
    for i in range(0, len(all_chunks), BATCH):
        batch = all_chunks[i:i + BATCH]
        collection.add(
            ids=[c["id"] for c in batch],
            documents=[c["text"] for c in batch],
            metadatas=[{"source": c["source"], "heading": c["heading"]} for c in batch],
        )
    return collection.count()


class Retriever:
    """Vyhledávač relevantních pasáží v hotovém indexu."""

    def __init__(self, collection) -> None:
        self._collection = collection

    @property
    def count(self) -> int:
        return self._collection.count()

    def retrieve(self, query: str, k: int = 4) -> List[Dict]:
        """Vrátí k nejpodobnějších pasáží k dotazu ``query``."""
        if not query or self._collection.count() == 0:
            return []
        result = self._collection.query(query_texts=[query], n_results=min(k, self._collection.count()))
        documents = result.get("documents", [[]])[0]
        metadatas = result.get("metadatas", [[]])[0]
        distances = result.get("distances", [[]])[0]
        out: List[Dict] = []
        for doc, meta, dist in zip(documents, metadatas, distances):
            out.append({
                "text": doc,
                "source": (meta or {}).get("source", ""),
                "heading": (meta or {}).get("heading", ""),
                "distance": dist,
            })
        return out


def load_retriever() -> Optional[Retriever]:
    """Načte hotový index. Pokud v DB nic není, vrátí None (aplikace funguje i bez RAG)."""
    if not DB_PATH.exists():
        return None
    try:
        client = _client()
        collection = client.get_collection(
            name=COLLECTION_NAME,
            embedding_function=_embedding_function(),
        )
    except Exception:
        return None
    if collection.count() == 0:
        return None
    return Retriever(collection)


def retrieve(query: str, k: int = 4) -> List[Dict]:
    """Zkratka: načte index a vrátí pasáže k dotazu (prázdné, když index chybí)."""
    retriever = load_retriever()
    if retriever is None:
        return []
    return retriever.retrieve(query, k=k)


def format_context(passages: List[Dict]) -> str:
    """Zformátuje pasáže do jednoho textového bloku pro systémový prompt."""
    if not passages:
        return ""
    parts = []
    for i, p in enumerate(passages, 1):
        label = p.get("heading") or p.get("source", "přednáška")
        parts.append(f"[{i}] {label}\n{p['text']}")
    return "\n\n".join(parts)


In [ ]:
%%writefile build_index.py
"""Skript pro sestavení vektorového indexu (znalostní báze).

Spuštění z kořene projektu:

    uv run python -m src.build_index          # postaví index, pokud neexistuje
    uv run python -m src.build_index --rebuild  # přegeneruje od nuly

Před spuštěním musí být v souboru ``.env`` nastaven ``OPENAI_API_KEY``
(embeddingy se počítají přes OpenAI).
"""

from __future__ import annotations

import argparse

from dotenv import load_dotenv

from src.rag import build_index


def main() -> None:
    load_dotenv()
    parser = argparse.ArgumentParser(description="Sestaví RAG index z přednášek.")
    parser.add_argument("--rebuild", action="store_true", help="Smazat a přegenerovat index.")
    args = parser.parse_args()

    pocet = build_index(rebuild=args.rebuild)
    print(f"Hotovo. V indexu je {pocet} pasáží.")


if __name__ == "__main__":
    main()


In [ ]:
# Sestavení / přegenerování vektorového indexu z přednášek (data/*.md)
# Vyžaduje OPENAI_API_KEY v .env (embeddingy). Běží pár desítek sekund.
from dotenv import load_dotenv
load_dotenv()

from rag import build_index

pocet = build_index(rebuild=True)
print(f'Index obsahuje {pocet} pasáží.')

In [4]:
%%writefile ../app.py
import os
import json
import streamlit as st
from dotenv import load_dotenv
from openai import OpenAI
from src.math_core import verify_equivalence, compute
from src.curriculum import CURRICULUM
from src.rag import load_retriever, format_context

# Načte OPENAI_API_KEY ze souboru .env v kořenové složce
load_dotenv()

st.set_page_config(page_title="AI Tutor Matematické analýzy – FSI VUT", page_icon="∫", layout="wide")

@st.cache_resource
def get_openai_client():
    return OpenAI()

@st.cache_resource
def get_retriever():
    """Načte RAG index z data/db (nebo None, pokud index neexistuje)."""
    return load_retriever()

client = get_openai_client()
retriever = get_retriever()

# --- Inicializace stavu konverzace ---
if "messages" not in st.session_state:
    st.session_state.messages = []
if "tool_logs" not in st.session_state:
    st.session_state.tool_logs = {}
if "sources" not in st.session_state:
    st.session_state.sources = {}

def reset_conversation():
    """Vymaže historii chatu, protokoly ověření i použité zdroje."""
    st.session_state.messages = []
    st.session_state.tool_logs = {}
    st.session_state.sources = {}

# --- Postranní panel (ovládání režimů, kapitoly a obtížnosti) ---
with st.sidebar:
    st.header("Nastavení výuky")
    mode = st.radio(
        "Režim asistenta:",
        ["Učitel (výklad teorie)", "Zkoušející (procvičování)"],
    )

    chapter = st.selectbox("Kapitola:", list(CURRICULUM.keys()))
    difficulty = st.select_slider(
        "Obtížnost úloh:",
        options=["Základní", "Střední (cvičení)", "Stačí umět všechno"],
        value="Střední (cvičení)",
    )

    st.divider()
    if st.button("🆕 Nová konverzace", use_container_width=True):
        reset_conversation()

    st.divider()
    if retriever is None:
        st.warning(
            "Znalostní báze (RAG) není k dispozici. Sestav ji příkazem:\n\n"
            "`uv run python -m src.build_index`",
            icon="📚",
        )
    else:
        st.caption(f"📚 Znalostní báze: **{retriever.count}** pasáží z přednášek")

# Když student změní kapitolu / režim / obtížnost, začneme novou konverzaci.
# Jinak by model dál viděl historii o starém tématu a bavil se o něm.
selection = (mode, chapter, difficulty)
if st.session_state.get("last_selection") != selection:
    if st.session_state.get("last_selection") is not None:
        reset_conversation()
        st.toast("Nastavení se změnilo – začínáme novou konverzaci.", icon="🔄")
    st.session_state.last_selection = selection

st.title("AI Tutor Matematické analýzy (FSI VUT)")
st.caption(f"Aktivní režim: **{mode}** | Kapitola: **{chapter}** | Obtížnost: **{difficulty}**")

# --- Definice nástrojů ze src/math_core.py pro OpenAI ---
tools = [
    {
        "type": "function",
        "function": {
            "name": "verify_equivalence",
            "description": "Ověří pomocí knihovny SymPy, zda jsou dva matematické výrazy algebraicky ekvivalentní.",
            "parameters": {
                "type": "object",
                "properties": {
                    "student_expr": {"type": "string", "description": "Výraz studenta v syntaxi SymPy (např. 'cos(2*x)')"},
                    "expected_expr": {"type": "string", "description": "Správný referenční výraz v syntaxi SymPy (např. '2*cos(2*x)')"},
                    "variable": {"type": "string", "description": "Proměnná, výchozí 'x'"}
                },
                "required": ["student_expr", "expected_expr"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "compute",
            "description": (
                "Přímo spočítá symbolický výsledek pomocí SymPy (nikdy nepočítej zpaměti). "
                "Použij pro derivaci, integrál, limitu, zjednodušení, rozvinutí, faktorizaci nebo řešení rovnice."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expr": {"type": "string", "description": "Vstupní výraz v syntaxi SymPy (např. 'sin(2*x)', 'x**2 - 1')"},
                    "operation": {
                        "type": "string",
                        "enum": ["simplify", "derivative", "derivative2", "integrate", "limit", "expand", "factor", "solve"],
                        "description": "Požadovaná operace."
                    },
                    "variable": {"type": "string", "description": "Proměnná výpočtu, výchozí 'x'"},
                    "point": {"type": "string", "description": "Bod pro limitu (např. '0', '2', 'oo', '-oo'). Povinné jen pro operation='limit'."}
                },
                "required": ["expr", "operation"]
            }
        }
    }
]

# Mapování názvů nástrojů na jejich lokální implementace
AVAILABLE_FUNCTIONS = {
    "verify_equivalence": verify_equivalence,
    "compute": compute,
}

# Doplnění konkrétní úlohy ke zvolené obtížnosti
DIFFICULTY_HINTS = {
    "Základní": "Zadávej jednoduché úlohy na přímé pochopení pojmu (jeden krok).",
    "Střední (cvičení)": "Zadávej typové úlohy na úrovni cvičení (několik kroků).",
    "Stačí umět všechno": "Zadávej i netriviální a zkouškové úlohy kombinující více kroků a souvislostí.",
}

def get_system_prompt(selected_mode: str, selected_chapter: str, selected_difficulty: str) -> str:
    base = (
        "Jsi specializovaný AI tutor Matematické analýzy pro studenty FSI VUT v Brně.\n"
        f"Aktuální okruh učiva: {selected_chapter}.\n"
        f"Požadovaná obtížnost: {selected_difficulty}. "
        f"{DIFFICULTY_HINTS.get(selected_difficulty, '')}\n\n"
        "Obecná pravidla:\n"
        "- Odpovídej vždy česky, odborně, ale srozumitelně.\n"
        "- Matematiku zapisuj v LaTeXu: $...$ pro řádkové vzorce a $$...$$ pro samostatné bloky.\n"
        "- Nikdy nepočítej zpaměti. Výsledky počítání získávej voláním nástroje `compute` "
        "(derivace, integrály, limity, úpravy) a správnost výrazů (např. odpovědi studenta) "
        "ověřuj nástrojem `verify_equivalence`.\n"
        "- Pokud si u definice nebo věty nejsi jistý, přiznej to a nevymýšlej si.\n"
        f"- Pokud se dotaz týká jiné kapitoly než „{selected_chapter}“, krátce odpověz a nabídni přepnutí kapitoly.\n"
    )
    if "Učitel" in selected_mode:
        return base + (
            "\nJsi v režimu UČITEL. Cílem je studentovi látku vysvětlit.\n"
            "Postupuj takto:\n"
            "1. Úvodem v 1–2 větách shrň, co daný pojem znamená a k čemu je.\n"
            "2. Uveď přesnou definici / větu (formálně, v LaTeXu), jak se probírá na přednáškách FSI.\n"
            "3. Vysvětli ji krok za krokem a přidej intuici „proč to platí“.\n"
            "4. Ukaž jeden řešený příklad a každý krok okomentuj.\n"
            "5. Upozorni na 2–3 nejčastější chyby studentů u tohoto typu úlohy.\n"
            "Vysvětluj maximálně srozumitelně a trpělivě:\n"
            "- žádný krok nevynechávej a nepředpokládej znalosti, které student mít nemusí,\n"
            "- každý netriviální pojem a krok znovu objasni, klidně i více způsoby (formálně i názorně),\n"
            "- používej jednoduchý jazyk a konkrétní příklady, ať student látku opravdu pochopí.\n"
            "Neptej se studenta na žádné kontrolní ani doplňující otázky. Nezadávej mu nic k dopočítání "
            "a nečekej na jeho odpověď – odpověz rovnou, úplně a do konce.\n"
            "Drž se jednoho pojmu a nezahlcuj – další témata můžeš nabídnout jako možný navazující krok."
        )
    return base + (
        "\nJsi v režimu ZKOUŠEJÍCÍ. Cílem je studenta procvičit.\n"
        "Pravidla:\n"
        "- Vždy zadej právě JEDEN příklad a pak čekej na studentovo řešení. Nezadávej více příkladů najednou.\n"
        "- Příklady tvoř jako obměny typových úloh z přednášek a cvičení (jiné koeficienty, jiné zadání); "
        "obtížnost odpovídá zvolené úrovni.\n"
        "- Než příklad zadáš, spočítej si správný referenční výsledek voláním `compute`.\n"
        "- Když student pošle výsledek nebo mezivýpočet, VŽDY ho ověř nástrojem `verify_equivalence` "
        "(student_expr vs. expected_expr).\n"
        "- Nikdy neprozrazuj celý výsledek dopředu; veď studenta nápovědami.\n"
        "- Při chybě urči její typ (např. chyba ve znaménku, chybná metoda integrace, chyba v algebraické úpravě, "
        "špatný definiční obor, záměna funkce a její hodnoty) a dej návodnou nápovědu.\n"
        "- Stupňuj nápovědu: 1) upozorni na oblast, 2) napověz pravidlo nebo větu, 3) ukaž jen první krok.\n"
        "- Když student vyřeší správně, krátce pochval a zadej další (klidně těžší) příklad.\n"
        "- Na konci můžeš shrnout, které typy úloh studentovi šly a které je potřeba docvičit."
    )

# Vykreslení historie zpráv
for idx, msg in enumerate(st.session_state.messages):
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if idx in st.session_state.sources:
            with st.expander("Použité pasáže z přednášek (RAG)"):
                for src in st.session_state.sources[idx]:
                    st.markdown(f"**{src['heading'] or src['source']}** — `{src['source']}`")
                    st.caption(src["text"][:400] + ("…" if len(src["text"]) > 400 else ""))
        if idx in st.session_state.tool_logs:
            with st.expander("Ověření výpočtu (SymPy)"):
                for log in st.session_state.tool_logs[idx]:
                    st.json(log)

# --- Zpracování nového dotazu ---
if prompt := st.chat_input("Napiš dotaz k teorii, požádej o příklad nebo zadej své řešení..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # RAG: vyhledá relevantní pasáže z přednášek
    passages = retriever.retrieve(prompt, k=4) if retriever else []

    api_messages = [{"role": "system", "content": get_system_prompt(mode, chapter, difficulty)}]
    if passages:
        api_messages.append({
            "role": "system",
            "content": (
                "Níže jsou relevantní pasáže z přednášek FSI k dotazu studenta. "
                "Vycházej především z nich a drž se jejich značení a postupů. "
                "Pokud odpověď v těchto pasážích není, řekni to a odpověz obecně.\n\n"
                + format_context(passages)
            ),
        })
    api_messages += st.session_state.messages

    with st.chat_message("assistant"):
        with st.spinner("Přemýšlím a ověřuji výpočty..."):
            verification_logs = []
            MAX_TOOL_ROUNDS = 5  # ochrana proti zacyklení volání nástroje

            for _ in range(MAX_TOOL_ROUNDS):
                response = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=api_messages,
                    tools=tools,
                )
                msg = response.choices[0].message

                # Model nevolá nástroj -> máme finální text
                if not msg.tool_calls:
                    break

                # Model zavolal některý nástroj ze SymPy -> vykonáme jej a pošleme výsledek zpět
                api_messages.append(msg)
                for tool_call in msg.tool_calls:
                    name = tool_call.function.name
                    args = json.loads(tool_call.function.arguments)
                    func = AVAILABLE_FUNCTIONS.get(name)
                    if func is None:
                        res_dict = {"error": f"Neznámý nástroj '{name}'."}
                    else:
                        res_dict = func(**args)
                    verification_logs.append({"nastroj": name, "vstup": args, "vysledek_sympy": res_dict})
                    api_messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": json.dumps(res_dict)
                    })

            reply_text = msg.content or ""
            st.markdown(reply_text)
            if passages:
                with st.expander("Použité pasáže z přednášek (RAG)"):
                    for src in passages:
                        st.markdown(f"**{src['heading'] or src['source']}** — `{src['source']}`")
                        st.caption(src["text"][:400] + ("…" if len(src["text"]) > 400 else ""))
            for log in verification_logs:
                with st.expander("Ověření výpočtu (SymPy)"):
                    st.json(log)

    st.session_state.messages.append({"role": "assistant", "content": reply_text})
    if passages:
        st.session_state.sources[len(st.session_state.messages) - 1] = passages
    if verification_logs:
        st.session_state.tool_logs[len(st.session_state.messages) - 1] = verification_logs


Overwriting ../app.py


In [ ]:
!streamlit run ../app.py

⠙⠹Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: /content/SIS_Matematik/app.py
⠸⠼⠴Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://three-pumas-stay.loca.lt
